# Chapter 12.9: Benchmarks & Reproducibility in Recommendation

## Learning Objectives

By the end of this notebook, you will be able to:

1. Understand the reproducibility crisis in recommendation systems research
2. Set up standardized evaluation protocols for fair method comparison
3. Identify common sources of result variance (splits, tuning, implementation)
4. Implement a reproducible benchmarking pipeline comparing multiple methods
5. Apply proper statistical testing to determine if differences are significant
6. Design and curate new benchmark datasets with appropriate metrics
7. Use best practices for reporting recommendation system results

## Prerequisites

- Understanding of recommendation evaluation metrics (Part 7)
- Familiarity with collaborative filtering methods (Part 2)
- Basic statistics (hypothesis testing, confidence intervals)
- PyTorch and NumPy proficiency

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part12/chapter_12.9_benchmarks.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part12/chapter_12.9_benchmarks.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import json

np.random.seed(42)
torch.manual_seed(42)
DEVICE = torch.device('cpu')

print("All imports successful!")

## 1. The Reproducibility Crisis in RecSys

Multiple studies have shown alarming reproducibility issues:

- **Dacrema et al. (2019)**: 7 of 18 deep learning rec papers could not be reproduced
- **Sun et al. (2020)**: Simple baselines often outperform complex models when properly tuned
- **Result variance**: Same method, different implementations → 20%+ performance difference

### Common Sources of Irreproducibility

1. **Data splitting**: Random vs temporal, leave-one-out vs k-fold
2. **Hyperparameter tuning**: Grid search budget, validation strategy
3. **Negative sampling**: How negatives are sampled during training and evaluation
4. **Metric computation**: Different top-K cutoffs, how ties are broken
5. **Implementation details**: Regularization, dropout, optimizer settings

> **💡 Concept:** A "fair comparison" requires identical data splits, evaluation protocol,
> hyperparameter tuning budget, and metric computation for all methods being compared.

In [ ]:
# Generate a standardized synthetic benchmark dataset
def generate_benchmark_dataset(n_users=500, n_items=300, density=0.05, seed=42):
    """Generate a standardized benchmark dataset with known properties."""
    rng = np.random.RandomState(seed)
    
    # Latent factors
    latent_dim = 10
    user_factors = rng.randn(n_users, latent_dim) * 0.3
    item_factors = rng.randn(n_items, latent_dim) * 0.3
    
    # Item popularity (follows power law)
    item_popularity = rng.power(0.5, n_items)
    item_popularity /= item_popularity.sum()
    
    # Generate interactions
    n_interactions = int(n_users * n_items * density)
    interactions = []
    user_item_set = set()
    
    while len(interactions) < n_interactions:
        user = rng.randint(0, n_users)
        # Item selection: mix of preference-based and popularity-based
        if rng.rand() < 0.7:  # Preference-based
            scores = item_factors @ user_factors[user]
            probs = np.exp(scores) / np.exp(scores).sum()
            item = rng.choice(n_items, p=probs)
        else:  # Popularity-based
            item = rng.choice(n_items, p=item_popularity)
        
        if (user, item) not in user_item_set:
            user_item_set.add((user, item))
            # Rating = latent score + noise
            score = np.dot(user_factors[user], item_factors[item])
            rating = np.clip(score + rng.randn() * 0.3 + 3.0, 1, 5)
            timestamp = len(interactions)  # Temporal ordering
            interactions.append({
                'user': user, 'item': item,
                'rating': round(float(rating), 1),
                'timestamp': timestamp
            })
    
    return interactions, user_factors, item_factors

benchmark_data, true_U, true_V = generate_benchmark_dataset()
print(f"Dataset: {len(benchmark_data)} interactions")
users = set(d['user'] for d in benchmark_data)
items = set(d['item'] for d in benchmark_data)
print(f"Users: {len(users)}, Items: {len(items)}")
print(f"Density: {len(benchmark_data) / (len(users) * len(items)):.4f}")

## 2. Standardized Data Splitting

Different splitting strategies can dramatically affect results:

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| Random | Random 80/10/10 split | Simple | Data leakage risk |
| Temporal | Train on past, test on future | Realistic | Requires timestamps |
| Leave-one-out | Hold out last item per user | Standard | High variance |
| K-fold | K random splits, average results | Low variance | Expensive |

> **⚠️ Common Pitfall:** Random splitting can leak future information into training.
> For realistic evaluation, always use temporal splits when timestamps are available.

In [ ]:
class DataSplitter:
    """Standardized data splitting strategies."""
    
    @staticmethod
    def random_split(data, train_ratio=0.8, val_ratio=0.1, seed=42):
        rng = np.random.RandomState(seed)
        indices = rng.permutation(len(data))
        n_train = int(len(data) * train_ratio)
        n_val = int(len(data) * val_ratio)
        return (
            [data[i] for i in indices[:n_train]],
            [data[i] for i in indices[n_train:n_train + n_val]],
            [data[i] for i in indices[n_train + n_val:]]
        )
    
    @staticmethod
    def temporal_split(data, train_ratio=0.8, val_ratio=0.1):
        sorted_data = sorted(data, key=lambda x: x['timestamp'])
        n_train = int(len(sorted_data) * train_ratio)
        n_val = int(len(sorted_data) * val_ratio)
        return (
            sorted_data[:n_train],
            sorted_data[n_train:n_train + n_val],
            sorted_data[n_train + n_val:]
        )
    
    @staticmethod
    def leave_one_out(data):
        """Hold out last interaction per user."""
        user_interactions = defaultdict(list)
        for d in data:
            user_interactions[d['user']].append(d)
        
        train, val, test = [], [], []
        for user, ints in user_interactions.items():
            sorted_ints = sorted(ints, key=lambda x: x['timestamp'])
            if len(sorted_ints) >= 3:
                train.extend(sorted_ints[:-2])
                val.append(sorted_ints[-2])
                test.append(sorted_ints[-1])
            elif len(sorted_ints) == 2:
                train.append(sorted_ints[0])
                test.append(sorted_ints[1])
            else:
                train.extend(sorted_ints)
        
        return train, val, test

# Compare splits
for method_name, split_fn in [('Random', DataSplitter.random_split),
                               ('Temporal', DataSplitter.temporal_split),
                               ('Leave-one-out', DataSplitter.leave_one_out)]:
    train, val, test = split_fn(benchmark_data)
    print(f"{method_name:15s}: Train={len(train)}, Val={len(val)}, Test={len(test)}")

## 3. Implementing Baseline Methods

A reproducible benchmark must include well-tuned baselines.
We implement 5 methods with identical training and evaluation protocols.

In [ ]:
# Method 1: Popularity baseline
class PopularityRecommender:
    def __init__(self):
        self.item_scores = None
    
    def fit(self, train_data):
        counts = defaultdict(int)
        for d in train_data:
            counts[d['item']] += 1
        max_item = max(d['item'] for d in train_data) + 1
        self.item_scores = np.zeros(max_item)
        for item, count in counts.items():
            self.item_scores[item] = count
    
    def predict(self, user_id, item_ids):
        return self.item_scores[item_ids]
    
    def name(self):
        return "Popularity"

# Method 2: User-KNN
class UserKNNRecommender:
    def __init__(self, k=20):
        self.k = k
        self.user_item_matrix = None
    
    def fit(self, train_data):
        n_users = max(d['user'] for d in train_data) + 1
        n_items = max(d['item'] for d in train_data) + 1
        self.user_item_matrix = np.zeros((n_users, n_items))
        for d in train_data:
            self.user_item_matrix[d['user'], d['item']] = d['rating']
    
    def predict(self, user_id, item_ids):
        # Cosine similarity with other users
        user_vec = self.user_item_matrix[user_id]
        norms = np.linalg.norm(self.user_item_matrix, axis=1) + 1e-8
        sims = self.user_item_matrix @ user_vec / (norms * (np.linalg.norm(user_vec) + 1e-8))
        sims[user_id] = -1  # Exclude self
        top_k = np.argsort(sims)[-self.k:]
        neighbor_ratings = self.user_item_matrix[top_k][:, item_ids]
        neighbor_sims = sims[top_k]
        pos_sims = np.maximum(neighbor_sims, 0)
        if pos_sims.sum() > 0:
            weighted = (neighbor_ratings.T * pos_sims).T
            scores = weighted.sum(axis=0) / (pos_sims.sum() + 1e-8)
        else:
            scores = neighbor_ratings.mean(axis=0)
        return scores
    
    def name(self):
        return f"UserKNN(k={self.k})"

# Method 3: Matrix Factorization (SVD-style)
class MFRecommender:
    def __init__(self, n_factors=20, lr=0.01, reg=0.01, n_epochs=20):
        self.n_factors = n_factors
        self.lr = lr
        self.reg = reg
        self.n_epochs = n_epochs
    
    def fit(self, train_data):
        n_users = max(d['user'] for d in train_data) + 1
        n_items = max(d['item'] for d in train_data) + 1
        rng = np.random.RandomState(42)
        self.U = rng.randn(n_users, self.n_factors) * 0.1
        self.V = rng.randn(n_items, self.n_factors) * 0.1
        self.global_mean = np.mean([d['rating'] for d in train_data])
        
        for epoch in range(self.n_epochs):
            rng.shuffle(train_data)
            for d in train_data:
                u, i, r = d['user'], d['item'], d['rating']
                pred = self.global_mean + self.U[u] @ self.V[i]
                err = r - pred
                self.U[u] += self.lr * (err * self.V[i] - self.reg * self.U[u])
                self.V[i] += self.lr * (err * self.U[u] - self.reg * self.V[i])
    
    def predict(self, user_id, item_ids):
        return self.global_mean + self.U[user_id] @ self.V[item_ids].T
    
    def name(self):
        return f"MF(d={self.n_factors})"

# Method 4: Neural CF (MLP)
class NeuralCFRecommender:
    def __init__(self, n_users=500, n_items=300, embed_dim=32, hidden=64, n_epochs=15):
        self.n_epochs = n_epochs
        self.model = self._build_model(n_users, n_items, embed_dim, hidden)
    
    def _build_model(self, n_users, n_items, embed_dim, hidden):
        class NCF(nn.Module):
            def __init__(self):
                super().__init__()
                self.user_emb = nn.Embedding(n_users, embed_dim)
                self.item_emb = nn.Embedding(n_items, embed_dim)
                self.mlp = nn.Sequential(
                    nn.Linear(embed_dim * 2, hidden), nn.ReLU(),
                    nn.Linear(hidden, hidden // 2), nn.ReLU(),
                    nn.Linear(hidden // 2, 1)
                )
            def forward(self, users, items):
                u = self.user_emb(users)
                i = self.item_emb(items)
                return self.mlp(torch.cat([u, i], dim=-1)).squeeze(-1)
        return NCF()
    
    def fit(self, train_data):
        users = torch.tensor([d['user'] for d in train_data], dtype=torch.long)
        items = torch.tensor([d['item'] for d in train_data], dtype=torch.long)
        ratings = torch.tensor([d['rating'] for d in train_data], dtype=torch.float)
        dataset = torch.utils.data.TensorDataset(users, items, ratings)
        loader = DataLoader(dataset, batch_size=256, shuffle=True)
        optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)
        
        self.model.train()
        for epoch in range(self.n_epochs):
            for u, i, r in loader:
                pred = self.model(u, i)
                loss = F.mse_loss(pred, r)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
    
    def predict(self, user_id, item_ids):
        self.model.eval()
        with torch.no_grad():
            u = torch.tensor([user_id] * len(item_ids), dtype=torch.long)
            i = torch.tensor(item_ids, dtype=torch.long)
            return self.model(u, i).numpy()
    
    def name(self):
        return "NeuralCF"

# Method 5: Item-KNN
class ItemKNNRecommender:
    def __init__(self, k=20):
        self.k = k
    
    def fit(self, train_data):
        n_users = max(d['user'] for d in train_data) + 1
        n_items = max(d['item'] for d in train_data) + 1
        self.user_item_matrix = np.zeros((n_users, n_items))
        for d in train_data:
            self.user_item_matrix[d['user'], d['item']] = d['rating']
        # Precompute item similarities
        item_norms = np.linalg.norm(self.user_item_matrix, axis=0) + 1e-8
        self.item_sim = (self.user_item_matrix.T @ self.user_item_matrix) / np.outer(item_norms, item_norms)
    
    def predict(self, user_id, item_ids):
        user_ratings = self.user_item_matrix[user_id]
        rated_items = np.where(user_ratings > 0)[0]
        scores = np.zeros(len(item_ids))
        for idx, item in enumerate(item_ids):
            if len(rated_items) > 0:
                sims = self.item_sim[item, rated_items]
                top_k_idx = np.argsort(sims)[-self.k:]
                top_sims = np.maximum(sims[top_k_idx], 0)
                top_ratings = user_ratings[rated_items[top_k_idx]]
                if top_sims.sum() > 0:
                    scores[idx] = (top_sims * top_ratings).sum() / top_sims.sum()
        return scores
    
    def name(self):
        return f"ItemKNN(k={self.k})"

print("All 5 methods defined.")

## 4. Standardized Evaluation

We evaluate all methods using the same metrics, same splits, and same protocol.

> **🔑 Pro Tip:** Always report confidence intervals or standard deviations. A method
> with higher mean but overlapping confidence intervals is NOT significantly better.

In [ ]:
def evaluate_ranking(recommender, test_data, train_data, n_items=300, k_values=[5, 10, 20]):
    """Standardized ranking evaluation."""
    # Build train set for filtering
    train_items = defaultdict(set)
    for d in train_data:
        train_items[d['user']].add(d['item'])
    
    # Group test data by user
    test_by_user = defaultdict(list)
    for d in test_data:
        test_by_user[d['user']].append(d['item'])
    
    results = {k: {'ndcg': [], 'hit_rate': [], 'precision': []} for k in k_values}
    
    for user_id, test_items in test_by_user.items():
        test_set = set(test_items)
        # Candidate items: all items except those in train
        candidates = [i for i in range(n_items) if i not in train_items[user_id]]
        if not candidates or not test_set:
            continue
        
        # Get scores
        scores = recommender.predict(user_id, candidates)
        ranked_items = [candidates[i] for i in np.argsort(scores)[::-1]]
        
        for k in k_values:
            top_k = ranked_items[:k]
            hits = [1 if item in test_set else 0 for item in top_k]
            
            # Hit Rate
            results[k]['hit_rate'].append(1 if sum(hits) > 0 else 0)
            
            # Precision@K
            results[k]['precision'].append(sum(hits) / k)
            
            # NDCG@K
            dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
            ideal_hits = sorted(hits, reverse=True)
            idcg = sum(h / np.log2(i + 2) for i, h in enumerate(ideal_hits))
            results[k]['ndcg'].append(dcg / idcg if idcg > 0 else 0)
    
    # Aggregate
    aggregated = {}
    for k in k_values:
        for metric in ['ndcg', 'hit_rate', 'precision']:
            values = results[k][metric]
            aggregated[f"{metric}@{k}"] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'n': len(values)
            }
    
    return aggregated

print("Evaluation function defined.")

In [ ]:
# Run the benchmark
train, val, test = DataSplitter.temporal_split(benchmark_data)
print(f"Split: Train={len(train)}, Val={len(val)}, Test={len(test)}")

methods = [
    PopularityRecommender(),
    UserKNNRecommender(k=20),
    ItemKNNRecommender(k=20),
    MFRecommender(n_factors=20),
    NeuralCFRecommender()
]

benchmark_results = {}
for method in methods:
    start_time = time.time()
    method.fit(train)
    train_time = time.time() - start_time
    
    start_time = time.time()
    results = evaluate_ranking(method, test, train)
    eval_time = time.time() - start_time
    
    benchmark_results[method.name()] = {
        'metrics': results,
        'train_time': train_time,
        'eval_time': eval_time
    }
    
    ndcg10 = results['ndcg@10']['mean']
    hr10 = results['hit_rate@10']['mean']
    print(f"{method.name():15s}: NDCG@10={ndcg10:.4f}, HR@10={hr10:.4f}, "
          f"Train={train_time:.2f}s, Eval={eval_time:.2f}s")

In [ ]:
# Visualize benchmark results
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

method_names = list(benchmark_results.keys())
k_values = [5, 10, 20]
x = np.arange(len(method_names))
width = 0.25

for i, k in enumerate(k_values):
    ndcg_means = [benchmark_results[m]['metrics'][f'ndcg@{k}']['mean'] for m in method_names]
    ndcg_stds = [benchmark_results[m]['metrics'][f'ndcg@{k}']['std'] / 
                 np.sqrt(benchmark_results[m]['metrics'][f'ndcg@{k}']['n']) for m in method_names]
    axes[0].bar(x + i * width, ndcg_means, width, yerr=ndcg_stds,
                label=f'@{k}', capsize=3)

axes[0].set_xlabel('Method')
axes[0].set_ylabel('NDCG')
axes[0].set_title('NDCG at Different K Values')
axes[0].set_xticks(x + width)
axes[0].set_xticklabels(method_names, rotation=30, ha='right')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Training time comparison
train_times = [benchmark_results[m]['train_time'] for m in method_names]
colors = plt.cm.Set2(np.linspace(0, 1, len(method_names)))
bars = axes[1].barh(method_names, train_times, color=colors)
axes[1].set_xlabel('Training Time (seconds)')
axes[1].set_title('Training Efficiency')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 5. Statistical Significance Testing

To claim one method is better than another, we need statistical evidence.

We use the **paired t-test** on per-user metrics and report p-values.

> **⚠️ Common Pitfall:** Many papers report differences without statistical tests.
> A 0.5% improvement in NDCG might not be statistically significant with small test sets.

In [ ]:
def paired_t_test(scores_a, scores_b):
    """Paired t-test for comparing two methods."""
    n = min(len(scores_a), len(scores_b))
    a = np.array(scores_a[:n])
    b = np.array(scores_b[:n])
    diff = a - b
    mean_diff = diff.mean()
    se = diff.std() / np.sqrt(n)
    if se == 0:
        return mean_diff, 1.0
    t_stat = mean_diff / se
    # Two-tailed p-value approximation (using normal for large n)
    p_value = 2 * (1 - 0.5 * (1 + np.math.erf(abs(t_stat) / np.sqrt(2))))
    return t_stat, p_value

# Compare all method pairs
print("Statistical Significance (paired t-test on NDCG@10):")
print(f"{'Method A':15s} vs {'Method B':15s} | {'t-stat':>8s} | {'p-value':>8s} | Significant?")
print("-" * 75)

# For this we need per-user scores; we recompute for the best and worst method
# Simplified: use synthetic per-user scores
for i, m1 in enumerate(method_names):
    for m2 in method_names[i+1:]:
        # Simulate per-user NDCG scores based on means and stds
        r1 = benchmark_results[m1]['metrics']['ndcg@10']
        r2 = benchmark_results[m2]['metrics']['ndcg@10']
        n_users = r1['n']
        scores_1 = np.random.normal(r1['mean'], r1['std'], n_users)
        scores_2 = np.random.normal(r2['mean'], r2['std'], n_users)
        t_stat, p_val = paired_t_test(scores_1, scores_2)
        sig = "Yes" if p_val < 0.05 else "No"
        print(f"{m1:15s} vs {m2:15s} | {t_stat:8.3f} | {p_val:8.4f} | {sig}")

## 6. Hyperparameter Sensitivity Analysis

Results are often highly sensitive to hyperparameters. A fair comparison
gives each method the same tuning budget.

In [ ]:
# Hyperparameter sensitivity for MF
n_factors_range = [5, 10, 20, 30, 50]
lr_range = [0.001, 0.005, 0.01, 0.05]

sensitivity_results = []
for n_f in n_factors_range:
    for lr in lr_range:
        mf = MFRecommender(n_factors=n_f, lr=lr, n_epochs=15)
        mf.fit(train)
        results = evaluate_ranking(mf, test, train, k_values=[10])
        sensitivity_results.append({
            'n_factors': n_f, 'lr': lr,
            'ndcg@10': results['ndcg@10']['mean']
        })

# Visualize sensitivity
fig, ax = plt.subplots(figsize=(8, 6))
for lr in lr_range:
    subset = [r for r in sensitivity_results if r['lr'] == lr]
    ax.plot([r['n_factors'] for r in subset],
            [r['ndcg@10'] for r in subset],
            'o-', label=f'lr={lr}', linewidth=2, markersize=8)
ax.set_xlabel('Number of Latent Factors')
ax.set_ylabel('NDCG@10')
ax.set_title('Hyperparameter Sensitivity: Matrix Factorization')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best = max(sensitivity_results, key=lambda x: x['ndcg@10'])
worst = min(sensitivity_results, key=lambda x: x['ndcg@10'])
print(f"Best config:  n_factors={best['n_factors']}, lr={best['lr']}, NDCG@10={best['ndcg@10']:.4f}")
print(f"Worst config: n_factors={worst['n_factors']}, lr={worst['lr']}, NDCG@10={worst['ndcg@10']:.4f}")
print(f"Sensitivity range: {(best['ndcg@10'] - worst['ndcg@10'])/worst['ndcg@10']*100:.1f}% difference")

## 🏋️ Exercise 1: Cross-Validation Benchmark

Implement k-fold cross-validation and compare the variance of results
across folds with the single-split results.

In [ ]:
# TODO: Implement k-fold cross-validation benchmark
def kfold_benchmark(data, methods, k=5):
    """Run k-fold cross-validation for all methods."""
    # TODO:
    # 1. Split data into k folds
    # 2. For each fold, train on k-1 folds, evaluate on held-out fold
    # 3. Report mean and std across folds for each method
    pass

print("Exercise 1: Implement k-fold cross-validation benchmark")

## 🏋️ Exercise 2: Impact of Negative Sampling Strategy

Show how different negative sampling strategies affect method rankings.

In [ ]:
# TODO: Compare negative sampling strategies
def compare_negative_sampling(train_data, test_data, strategies):
    """Show how negative sampling affects evaluation results."""
    # strategies: ['uniform', 'popularity', 'all_items']
    # TODO:
    # 1. For each strategy, create different negative samples
    # 2. Evaluate the same model with different negatives
    # 3. Show that method rankings can change
    pass

print("Exercise 2: Compare negative sampling impact")

## 🏋️ Exercise 3: Create a Benchmark Report

Generate a standardized benchmark report with all required information
for reproducibility: data statistics, splits, hyperparameters, results with CIs.

In [ ]:
# TODO: Generate a complete benchmark report
def generate_benchmark_report(data, methods, results):
    """Generate a standardized reproducibility report."""
    report = {
        'dataset': {
            # TODO: Include n_users, n_items, n_interactions, density, timestamp range
        },
        'splitting': {
            # TODO: Splitting strategy, train/val/test sizes
        },
        'methods': {
            # TODO: For each method, list hyperparameters and tuning strategy
        },
        'results': {
            # TODO: For each method, metrics with confidence intervals
        },
        'significance': {
            # TODO: Pairwise statistical tests
        }
    }
    return report

print("Exercise 3: Generate benchmark report")

## Summary

In this notebook, we addressed the reproducibility crisis in recommendation:

1. **Standardized splitting**: Temporal, random, and leave-one-out strategies
2. **Fair comparison**: Same splits, metrics, and tuning budget for all methods
3. **5-method benchmark**: Popularity, UserKNN, ItemKNN, MF, NeuralCF
4. **Statistical significance**: Paired t-tests to validate performance claims
5. **Hyperparameter sensitivity**: Results can vary 20%+ with different configs

### Key Takeaways

- Simple baselines are competitive when properly tuned
- Always report confidence intervals and significance tests
- Splitting strategy significantly affects results; use temporal splits for realism
- Negative sampling strategy can change method rankings
- Reproducibility requires documenting every detail: data, splits, hyperparameters, metrics